In [ ]:
import random
from collections import deque
from datetime import datetime

random.seed(7)
# ---------------------------------
# Reactor Environment
# ---------------------------------

class ReactorMonitor:
    def __init__(self, max_power=3000):
        self.max_power = max_power

        self.feedback_coefficient = 0.002
        self.heat_gain_factor = 0.006
        self.cooling_rate = 3.0

        self.reset()

    def reset(self):
        self.core_temperature = 230.0
        self.control_rod_position = 50.0
        self.thermal_power = 0.0

        return self.get_continuous_state()

    def get_continuous_state(self):
        return [
            self.core_temperature / 300.0,
            self.thermal_power / self.max_power,
            self.control_rod_position / 100.0
        ]

    def step(self, action):

        # 0 = Insert rods
        # 1 = Hold position
        # 2 = Extract rods

        if action == 0:
            self.control_rod_position -= 5

        elif action == 2:
            self.control_rod_position += 5

        self.control_rod_position = max(
            0.0,
            min(100.0, self.control_rod_position)
        )

        base_reactivity = (
            self.control_rod_position / 100.0
        )

        temperature_penalty = max(
            0.0,
            (self.core_temperature - 200.0)
            * self.feedback_coefficient
        )

        adjusted_reactivity = max(
            0.0,
            base_reactivity - temperature_penalty
        )

        self.thermal_power = min(
            self.max_power,
            adjusted_reactivity * self.max_power
        )

        self.core_temperature += (
            self.thermal_power
            * self.heat_gain_factor
        ) - self.cooling_rate

        self.core_temperature = max(
            200.0,
            self.core_temperature
        )

        reward = (
            10.0
            - ((self.core_temperature - 240.0) ** 2)
            * 0.1
        )

        done = False

        if self.core_temperature > 280.0:
            reward = -100.0
            done = True

        elif self.core_temperature < 210.0:
            reward = -100.0
            done = True

        next_state = (
            self.get_continuous_state()
        )

        return next_state, reward, done

    def display_status(self, action):

        print(
            f"Time: "
            f"{datetime.now().strftime('%H:%M:%S')}"
        )

        print(
            f"Temperature: "
            f"{self.core_temperature:.2f} °C"
        )

        print(
            f"Thermal Power: "
            f"{self.thermal_power:.1f} MW"
        )

        print(
            f"Control Rod Position: "
            f"{self.control_rod_position:.1f}"
        )

        print(
            f"Action: "
            f"{action_names[action]}"
        )

        print("-" * 50)


# ---------------------------------
# Tabular Continuous-State Agent
# ---------------------------------

class QAgent:
    def __init__(
        self,
        alpha=0.1,
        gamma=0.95,
        epsilon=0.2
    ):

        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        self.actions = [0, 1, 2]

        self.q_table = {}

        self.replay_memory = deque(
            maxlen=2000
        )

    def state_key(self, state):

        return tuple(
            round(value, 3)
            for value in state
        )

    def get_q_values(self, state):

        key = self.state_key(state)

        if key not in self.q_table:

            self.q_table[key] = {
                action: 0.0
                for action in self.actions
            }

        return self.q_table[key]

    def choose_action(self, state):

        if random.random() < self.epsilon:
            return random.choice(
                self.actions
            )

        q_values = self.get_q_values(
            state
        )

        return max(
            q_values,
            key=q_values.get
        )

    def store_experience(
        self,
        state,
        action,
        reward,
        next_state,
        done
    ):

        self.replay_memory.append(
            (
                state,
                action,
                reward,
                next_state,
                done
            )
        )

    def replay(self, batch_size=32):

        if (
            len(self.replay_memory)
            < batch_size
        ):
            return

        batch = random.sample(
            self.replay_memory,
            batch_size
        )

        for (
            state,
            action,
            reward,
            next_state,
            done
        ) in batch:

            q_values = self.get_q_values(
                state
            )

            old_q = q_values[action]

            next_q_values = (
                self.get_q_values(
                    next_state
                )
            )

            best_future_q = max(
                next_q_values.values()
            )

            target = reward

            if not done:
                target += (
                    self.gamma
                    * best_future_q
                )

            q_values[action] = (
                old_q
                + self.alpha
                * (target - old_q)
            )


# ---------------------------------
# Shared Constants
# ---------------------------------

action_names = {
    0: "Insert Rods",
    1: "Hold Position",
    2: "Extract Rods"
}


# ---------------------------------
# Main Training Phase
# ---------------------------------

env = ReactorMonitor()

agent = QAgent()

episodes = 500
steps_per_episode = 50

for episode in range(episodes):

    state = env.reset()

    for step in range(
        steps_per_episode
    ):

        action = (
            agent.choose_action(
                state
            )
        )

        (
            next_state,
            reward,
            done
        ) = env.step(action)

        agent.store_experience(
            state,
            action,
            reward,
            next_state,
            done
        )

        state = next_state

        if done:
            break

    # Replay once per episode
    agent.replay(
        batch_size=32
    )


# ---------------------------------
# Verification Run
# ---------------------------------

print(
    "MODULAR PRODUCTION RUN: "
    "STABLE ENVIRONMENT"
)

print("=" * 50)

agent.epsilon = 0.0

state = env.reset()

for step in range(15):

    action = agent.choose_action(
        state
    )

    (
        next_state,
        reward,
        done
    ) = env.step(action)

    env.display_status(action)

    print(
        f"Reward: {reward:.2f}"
    )

    print(
        f"Distance From 240°C: "
        f"{abs(env.core_temperature - 240.0):.2f} °C"
    )

    print("=" * 50)

    state = next_state

    if done:

        print(
            "SAFETY SHUTDOWN - "
            "EPISODE TERMINATED"
        )

        break
# ---------------------------------
# PHASE 2: ADVERSARIAL BOOTCAMP TRAINING
# ---------------------------------

print("\n--- PHASE 2: ADVERSARIAL BOOTCAMP TRAINING ---")

agent.epsilon = 0.2

bootcamp_episodes = 1000
steps_per_episode = 50

for episode in range(bootcamp_episodes):

    # Configure episode conditions FIRST
    if random.random() < 0.30:
        env.cooling_rate = 1.5
    else:
        env.cooling_rate = 3.0

    # Then initialize the episode
    state = env.reset()

    for step in range(steps_per_episode):

        action = agent.choose_action(state)

        next_state, reward, done = env.step(action)

        agent.store_experience(
            state,
            action,
            reward,
            next_state,
            done
        )

        state = next_state

        if done:
            break

    # Replay exactly once per episode
    agent.replay(batch_size=32)

# Restore nominal operating conditions
env.cooling_rate = 3.0


# ---------------------------------
# FINAL ADVERSARIAL VERIFICATION RUN
# ---------------------------------

print("\nFINAL ADVERSARIAL VERIFICATION RUN")
print("=" * 60)

agent.epsilon = 0.0

state = env.reset()

for step in range(15):

    if step == 5:
        print(
            "\n!!! EMERGENCY: COOLING PUMP FAILURE INJECTED !!!\n"
        )
        env.cooling_rate = 1.5

    action = agent.choose_action(state)

    next_state, reward, done = env.step(action)

    distance_from_target = abs(
        env.core_temperature - 240.0
    )

    print(f"Step: {step + 1}")

    print(
        f"Core Temperature: "
        f"{env.core_temperature:.2f} °C"
    )

    print(
        f"Thermal Power: "
        f"{env.thermal_power:.1f} MW"
    )

    print(
        f"Control Rod Position: "
        f"{env.control_rod_position:.1f}"
    )

    print(
        f"Action Executed: "
        f"{action_names[action]}"
    )

    print(
        f"Reward: {reward:.2f}"
    )

    print(
        f"Distance From Target: "
        f"{distance_from_target:.2f} °C"
    )

    print("-" * 60)

    if env.core_temperature > 280.0:
        print(
            "CATASTROPHIC MELTDOWN: "
            "AI SYSTEM OVERWHELM"
        )
        break

    state = next_state

    if done:
        print(
            "EPISODE TERMINATED "
            "BY SAFETY LIMITS"
        )
        break

MODULAR PRODUCTION RUN: STABLE ENVIRONMENT
Time: 14:43:16
Temperature: 234.02 °C
Thermal Power: 1170.0 MW
Control Rod Position: 45.0
Action: Insert Rods
--------------------------------------------------
Reward: 6.42
Distance From 240°C: 5.98 °C
Time: 14:43:16
Temperature: 237.00 °C
Thermal Power: 995.9 MW
Control Rod Position: 40.0
Action: Insert Rods
--------------------------------------------------
Reward: 9.10
Distance From 240°C: 3.00 °C
Time: 14:43:16
Temperature: 238.96 °C
Thermal Power: 828.0 MW
Control Rod Position: 35.0
Action: Insert Rods
--------------------------------------------------
Reward: 9.89
Distance From 240°C: 1.04 °C
Time: 14:43:16
Temperature: 239.96 °C
Thermal Power: 666.2 MW
Control Rod Position: 30.0
Action: Insert Rods
--------------------------------------------------
Reward: 10.00
Distance From 240°C: 0.04 °C
Time: 14:43:16
Temperature: 241.82 °C
Thermal Power: 810.2 MW
Control Rod Position: 35.0
Action: Extract Rods
-------------------------------------

In [ ]:
# ==================================================
# MODULE B: ADVERSARIAL STRESS-TEST LAB
# ==================================================

import random
from collections import deque
from datetime import datetime


class ReactorMonitor:
    def __init__(self, max_power=3000):
        self.max_power = max_power

        self.feedback_coefficient = 0.002
        self.heat_gain_factor = 0.006
        self.cooling_rate = 3.0

        self.reset()

    def reset(self):
        self.core_temperature = 230.0
        self.control_rod_position = 50.0
        self.thermal_power = 0.0

        return self.get_continuous_state()

    def get_continuous_state(self):
        return [
            self.core_temperature / 300.0,
            self.thermal_power / self.max_power,
            self.control_rod_position / 100.0
        ]

    def step(self, action):
        # Actions:
        # 0 = Insert rods
        # 1 = Hold position
        # 2 = Extract rods

        if action == 0:
            self.control_rod_position -= 5
        elif action == 2:
            self.control_rod_position += 5

        self.control_rod_position = max(
            0.0,
            min(100.0, self.control_rod_position)
        )

        base_reactivity = self.control_rod_position / 100.0

        temperature_penalty = max(
            0.0,
            (self.core_temperature - 200.0)
            * self.feedback_coefficient
        )

        adjusted_reactivity = max(
            0.0,
            base_reactivity - temperature_penalty
        )

        self.thermal_power = min(
            self.max_power,
            adjusted_reactivity * self.max_power
        )

        self.core_temperature += (
            self.thermal_power * self.heat_gain_factor
        ) - self.cooling_rate

        self.core_temperature = max(
            200.0,
            self.core_temperature
        )

        done = False

        reward = (
            10.0
            - ((self.core_temperature - 240.0) ** 2) * 0.1
        )

        # Safety limits
        if self.core_temperature > 280.0:
            reward = -100.0
            done = True

        elif self.core_temperature < 210.0:
            reward = -100.0
            done = True

        next_state = self.get_continuous_state()

        return next_state, reward, done

    def display_status(self, action):
        action_names = {
            0: "Insert Rods",
            1: "Hold Position",
            2: "Extract Rods"
        }

        print(
            f"Time: {datetime.now().strftime('%H:%M:%S')}"
        )
        print(
            f"Temperature: {self.core_temperature:.2f} °C"
        )
        print(
            f"Thermal Power: {self.thermal_power:.1f} MW"
        )
        print(
            f"Control Rod Position: "
            f"{self.control_rod_position:.1f}"
        )
        print(
            f"Action: {action_names[action]}"
        )
        print("-" * 45)


class QAgent:
    def __init__(
        self,
        alpha=0.1,
        gamma=0.95,
        epsilon=0.2
    ):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        self.actions = [0, 1, 2]

        self.q_table = {}

        self.replay_memory = deque(maxlen=2000)

    def state_key(self, state):
        return tuple(
            round(value, 3)
            for value in state
        )

    def get_q_values(self, state):
        key = self.state_key(state)

        if key not in self.q_table:
            self.q_table[key] = {
                action: 0.0
                for action in self.actions
            }

        return self.q_table[key]

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)

        q_values = self.get_q_values(state)

        return max(
            q_values,
            key=q_values.get
        )

    def store_experience(
        self,
        state,
        action,
        reward,
        next_state,
        done
    ):
        self.replay_memory.append(
            (
                state,
                action,
                reward,
                next_state,
                done
            )
        )

    def replay(self, batch_size=32):
        if len(self.replay_memory) < batch_size:
            return

        batch = random.sample(
            self.replay_memory,
            batch_size
        )

        for (
            state,
            action,
            reward,
            next_state,
            done
        ) in batch:

            q_values = self.get_q_values(state)

            old_q = q_values[action]

            next_q_values = self.get_q_values(
                next_state
            )

            best_future_q = max(
                next_q_values.values()
            )

            target = reward

            if not done:
                target += (
                    self.gamma
                    * best_future_q
                )

            q_values[action] = (
                old_q
                + self.alpha
                * (target - old_q)
            )


# ---------------------------------
# Training Phase
# ---------------------------------

env = ReactorMonitor()
agent = QAgent()

episodes = 500
steps_per_episode = 50

for episode in range(episodes):

    state = env.reset()

    for step in range(steps_per_episode):

        action = agent.choose_action(
            state
        )

        next_state, reward, done = (
            env.step(action)
        )

        agent.store_experience(
            state,
            action,
            reward,
            next_state,
            done
        )

        state = next_state

        if done:
            break

    # Replay runs once after the episode
    agent.replay(batch_size=32)

# ---------------------------------
# PHASE 2: ADVERSARIAL BOOTCAMP TRAINING
# ---------------------------------

print("\n--- PHASE 2: ADVERSARIAL BOOTCAMP TRAINING ---")

agent.epsilon = 0.2

bootcamp_episodes = 200
steps_per_episode = 50

for episode in range(bootcamp_episodes):

    # Configure episode conditions FIRST
    if random.random() < 0.30:
        env.cooling_rate = 1.5
    else:
        env.cooling_rate = 3.0

    # Then initialize the episode
    state = env.reset()

    for step in range(steps_per_episode):

        action = agent.choose_action(state)

        next_state, reward, done = env.step(action)

        agent.store_experience(
            state,
            action,
            reward,
            next_state,
            done
        )

        state = next_state

        if done:
            break

    # Replay exactly once per episode
    agent.replay(batch_size=32)

# Restore nominal operating conditions
env.cooling_rate = 3.0

def run_adversarial_test(trained_agent):
    # Create a fresh reactor environment
    test_env = ReactorMonitor()

    # Use trained policy only
    trained_agent.epsilon = 0.0

    state = test_env.reset()

    for step in range(15):

        # Inject malfunction at step 5
        if step == 5:
            print("\n!!! EMERGENCY: COOLING PUMP FAILURE INJECTED !!!\n")

        if step >= 5:
            test_env.cooling_rate = 1.5

        action = trained_agent.choose_action(state)

        next_state, reward, done = test_env.step(action)

        distance_from_target = abs(
            test_env.core_temperature - 240.0
        )

        action_names = {
            0: "Insert Rods",
            1: "Hold Position",
            2: "Extract Rods"
        }

        print(
            f"Time: {datetime.now().strftime('%H:%M:%S')}"
        )
        print(
            f"Step: {step + 1}"
        )
        print(
            f"Core Temperature: "
            f"{test_env.core_temperature:.2f} °C"
        )
        print(
            f"Action Executed: "
            f"{action_names[action]}"
        )
        print(
            f"Reward: {reward:.2f}"
        )
        print(
            f"Distance From Target: "
            f"{distance_from_target:.2f} °C"
        )
        print("-" * 50)

        # Catastrophic failure check
        if test_env.core_temperature > 280.0:
            print(
                "CATASTROPHIC MELTDOWN: "
                "AI SYSTEM OVERWHELM"
            )
            break

        state = next_state

        if done:
            print(
                "EPISODE TERMINATED BY "
                "REACTOR SAFETY LIMITS"
            )
            break


# Run the adversarial stress test using the
# already-trained agent from Cell 1

run_adversarial_test(agent)


--- PHASE 2: ADVERSARIAL BOOTCAMP TRAINING ---
Time: 08:45:41
Step: 1
Core Temperature: 234.02 °C
Action Executed: Insert Rods
Reward: 6.42
Distance From Target: 5.98 °C
--------------------------------------------------
Time: 08:45:41
Step: 2
Core Temperature: 237.00 °C
Action Executed: Insert Rods
Reward: 9.10
Distance From Target: 3.00 °C
--------------------------------------------------
Time: 08:45:41
Step: 3
Core Temperature: 238.96 °C
Action Executed: Insert Rods
Reward: 9.89
Distance From Target: 1.04 °C
--------------------------------------------------
Time: 08:45:41
Step: 4
Core Temperature: 239.96 °C
Action Executed: Insert Rods
Reward: 10.00
Distance From Target: 0.04 °C
--------------------------------------------------
Time: 08:45:41
Step: 5
Core Temperature: 241.82 °C
Action Executed: Extract Rods
Reward: 9.67
Distance From Target: 1.82 °C
--------------------------------------------------

!!! EMERGENCY: COOLING PUMP FAILURE INJECTED !!!

Time: 08:45:41
Step: 6
Core T